# Capstone Gate 4 (Part 2): Find the Top Reasons (Drivers)
Goal: for each athlete-day, find the 2 or 3 factors that explain why the tier is low.

How: we compare today's numbers with that athlete's **own usual level**.

In [1]:
# Import the libraries we need
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

## Part A: Prepare the Data and Load the Model

In [2]:
# Load the CSV file
df = pd.read_csv("athlete_recovery_synthetic.csv")


def get_tier(score):
    if score >= 75:
        return "Green"
    elif score >= 40:
        return "Orange"
    else:
        return "Red"

df["Tier"] = df["Recovery_Score"].apply(get_tier)
df["Sleep_Missing"] = df["Sleep_Duration_Hours"].isna().astype(int)
df["Stress_Num"] = df["Stress_Level"].map({"Low": 1, "Medium": 2, "High": 3})
df["Training_Load"] = df["Training_Duration_Min"] * df["Training_Intensity"]

# Make sure the days are in order for each athlete
df = df.sort_values(["Athlete_ID", "Day"]).reset_index(drop=True)

In [3]:
# Load the model we saved in Part 1
saved = joblib.load("readiness_model.joblib")
model = saved["model"]
features = saved["features"]
medians = saved["medians"]

In [4]:
# Same athlete split as before (same random_state = 42)
athlete_ids = df["Athlete_ID"].unique()
train_ids, test_ids = train_test_split(athlete_ids, test_size=0.2, random_state=42)

# We show the drivers for the TEST athletes
df_test = df[df["Athlete_ID"].isin(test_ids)].copy()

# Predict the tier for each test day
df_test["Predicted_Tier"] = model.predict(df_test[features].fillna(medians))
print(df_test["Predicted_Tier"].value_counts())

Predicted_Tier
Orange    744
Red       631
Green     299
Name: count, dtype: int64


## Part B: How Much Does Each Factor Push Toward Red?

The model gives each input a **weight** (we saw them in Part 1: HRV and sleep push away from Red, soreness and stress push toward Red).

For one day: **push toward Red = weight x how far today's value is from the team average.**

For example, a very low HRV has a big push toward Red. A positive push means "this factor is hurting the athlete today".

In [5]:
# Pull the weights and the scaler out of the saved model
scaler = model.named_steps["scale"]
logreg = model.named_steps["logreg"]

# Weight of each input for the Red tier
red_position = list(logreg.classes_).index("Red")
weights = pd.Series(logreg.coef_[red_position], index=features)

# Team average of each input (from the training data)
team_average = pd.Series(scaler.mean_, index=features)

# Today's values on the same scale (how far from the team average)
scaled = pd.DataFrame(scaler.transform(df_test[features].fillna(medians)),
                      columns=features, index=df_test.index)

# Push toward Red for every test day
push = scaled * weights

**Note:** we only report factors a coach understands: HRV, sleep, soreness, stress and resting heart rate. We also report when sleep was **not logged**, because the model uses that as a clue. We leave out the three training columns, because they overlap with each other and are hard to explain one by one.

## Part C: Pick the Top Drivers

In [6]:
# Nice names for the factors we report
names = {
    "HRV_ms": "HRV",
    "Sleep_Duration_Hours": "Sleep",
    "Muscle_Soreness": "Muscle soreness",
    "Stress_Num": "Stress (1 to 3)",
    "Resting_Heart_Rate": "Resting heart rate",
}

# Find the top drivers for one day (top 3 with a push of 0.5 or more)
def get_drivers(i, how_many=3, minimum=0.5):
    pushes = push.loc[i, list(names) + ["Sleep_Missing"]].sort_values(ascending=False)
    pushes = pushes[pushes >= minimum].head(how_many)

    texts = []
    for col in pushes.index:
        if col == "Sleep_Missing":
            texts.append("Sleep was not logged")
        else:
            value = df_test.loc[i, col]
            word = "low" if value < team_average[col] else "high"
            texts.append(f"{names[col]} is {word}: {value:.1f} vs team average {team_average[col]:.1f}")
    return texts

In [7]:
# Show 3 examples for each predicted tier
for tier in ["Red", "Orange", "Green"]:
    print("=====", tier, "=====")
    examples = df_test[df_test["Predicted_Tier"] == tier].sample(3, random_state=1)
    for i in examples.index:
        drivers = get_drivers(i)
        print("Athlete", df_test.loc[i, "Athlete_ID"], "| Day", df_test.loc[i, "Day"], "->",
              "; ".join(drivers) if drivers else "no clear driver")
    print()

===== Red =====
Athlete 1025 | Day 19 -> Stress (1 to 3) is high: 3.0 vs team average 2.0; Muscle soreness is high: 6.7 vs team average 5.1
Athlete 1148 | Day 2 -> Stress (1 to 3) is high: 3.0 vs team average 2.0; Sleep is low: 7.2 vs team average 7.5
Athlete 1148 | Day 5 -> HRV is low: 62.0 vs team average 73.9; Sleep is low: 6.8 vs team average 7.5; Muscle soreness is high: 8.1 vs team average 5.1

===== Orange =====
Athlete 1218 | Day 20 -> HRV is low: 66.0 vs team average 73.9
Athlete 1157 | Day 7 -> Sleep is low: 6.3 vs team average 7.5
Athlete 1007 | Day 15 -> HRV is low: 65.0 vs team average 73.9

===== Green =====
Athlete 1165 | Day 2 -> Sleep is low: 7.3 vs team average 7.5
Athlete 1286 | Day 28 -> no clear driver
Athlete 1063 | Day 13 -> no clear driver



**Answer:** *Do the reasons make sense?*

- **Red days** show clear, believable reasons: low HRV, low sleep, high soreness and high stress. For example, Athlete 1148 on Day 5 has HRV 62 (team average 74), sleep 6.8 (7.5) and soreness 8.1 (5.1).
- **Orange days** usually have one reason, and **Green days** usually have none. This is what we want.
- **Careful:** small differences can be reported too. For example, "Sleep is low: 7.2 vs team average 7.5" is only a small gap. If we only want bigger problems, we can raise the minimum push from 0.5 to a higher number.

## Part D: How Many Drivers Do We Find?

In [8]:
# Count the drivers for every test day
df_test["Driver_List"] = [get_drivers(i) for i in df_test.index]
df_test["Number_Of_Drivers"] = df_test["Driver_List"].apply(len)

# For each predicted tier, what percent of days have 0, 1, 2 or 3 drivers?
print((pd.crosstab(df_test["Predicted_Tier"], df_test["Number_Of_Drivers"], normalize="index") * 100).round(1))

Number_Of_Drivers     0     1     2     3
Predicted_Tier                           
Green              82.3  17.4   0.3   0.0
Orange             24.3  55.2  19.5   0.9
Red                 0.0   7.3  38.7  54.0


**Answer:** *Does every tier get reasons?*

- **Red:** every Red day has at least one reason, and 93% have two or three.
- **Orange:** about 76% have at least one reason, and 24% have none.
- **Green:** about 82% have no reason.
- The lower the tier, the more reasons we show. Orange days with no reason are borderline days where several small things add up. For those, the app will say "no single clear driver" and the fallback note will handle it.

## Part E: Check by Hand

Gate 4 asks us to check that the drivers are right. We take a few Red days and compare today's numbers with the athlete's usual numbers ourselves.

In [9]:
# Look at 2 Red days in detail
red_days = df_test[df_test["Predicted_Tier"] == "Red"].sample(2, random_state=5)

for i in red_days.index:
    print("Athlete", df_test.loc[i, "Athlete_ID"], "| Day", df_test.loc[i, "Day"], "| Real score:", df_test.loc[i, "Recovery_Score"])

    table = pd.DataFrame({
        "Today": [df_test.loc[i, col] for col in names],
        "Team average": [team_average[col] for col in names],
        "Push toward Red": [push.loc[i, col] for col in names],
    }, index=list(names.values())).round(1)
    print(table)
    print("Sleep logged?", "No" if df_test.loc[i, "Sleep_Missing"] == 1 else "Yes")
    print("Drivers found:", "; ".join(get_drivers(i)))
    print()

Athlete 1024 | Day 6 | Real score: 14.1
                    Today  Team average  Push toward Red
HRV                  49.0          73.9              5.7
Sleep                 7.6           7.5             -0.2
Muscle soreness       5.8           5.1              0.4
Stress (1 to 3)       2.0           2.0             -0.0
Resting heart rate   65.0          57.2              0.2
Sleep logged? Yes
Drivers found: HRV is low: 49.0 vs team average 73.9

Athlete 1289 | Day 10 | Real score: 14.2
                    Today  Team average  Push toward Red
HRV                  57.0          73.9              3.8
Sleep                 6.6           7.5              2.4
Muscle soreness       4.9           5.1             -0.1
Stress (1 to 3)       2.0           2.0             -0.0
Resting heart rate   61.0          57.2              0.1
Sleep logged? Yes
Drivers found: HRV is low: 57.0 vs team average 73.9; Sleep is low: 6.6 vs team average 7.5



**Answer:** *Are the drivers correct?*

- **Athlete 1024, Day 6:** HRV is 49 vs a team average of 74, which is very low. All other factors are close to average. So HRV is the only driver, and that is correct.
- **Athlete 1289, Day 10:** HRV is 57 vs 74 and sleep is 6.6 vs 7.5. Both are clearly low, and the notebook lists both.
- The drivers match the raw numbers.
- For the Gate 4 slide, check about 10 cases by hand. Change `random_state=5` to see different days.

## Part F: Save the Results

In [10]:
# Save a table for the next notebook (the note writer)
output = df_test[["Athlete_ID", "Day", "Tier", "Predicted_Tier", "Number_Of_Drivers"]].copy()
output["Drivers"] = df_test["Driver_List"].apply(lambda texts: "; ".join(texts))

output.to_csv("drivers_output.csv", index=False)
print("Saved drivers_output.csv with", len(output), "rows")

Saved drivers_output.csv with 1674 rows


## Summary (for the Gate 4 slide)

| Item | Result |
|---|---|
| Method | Push toward Red = the model's weight x how far today is from the team average. Show the top 3 with a push of 0.5 or more. |
| Why this method | It uses the model's own weights, so the reasons match what the model does. It is plain pandas, with no AI. |
| Factors reported | HRV, sleep, muscle soreness, stress, resting heart rate, and "sleep was not logged" |
| Reasons found (test days) | Red: 100% have at least one. Orange: 76%. Green: 18%. |
| What we tried first | Comparing with the athlete's own usual level. It missed too many days (21% of Red, 58% of Orange had no reason), so we replaced it. |
| Limitations | Compared with the team average, not the athlete's own usual level. Training columns left out. Small gaps can be reported. Synthetic data. |
| Output | drivers_output.csv, used by the next notebook |